# CHSH Game with 3‑Phase Signals + DSOGI + Bohmian‑Style “Pilot + Pointer” Story

This notebook is designed to be **digestable**:

- **Signals:** classical 3‑phase time‑domain waveforms  
- **Analysis box:** Clarke → DSOGI → phasor extraction  
- **Measurement score:** \(S = |a_x|^2 - |a_y|^2\)  
- **Pointer latch:** a bistable variable \(y(t)\) driven by the local score → **vote** \(A=\mathrm{sign}(y)\in\{\pm1\}\)  
- **CHSH correlator:** \(E(a,b)=\langle A\cdot B\rangle\) (strict)

Two source modes:
1) **Local-only**: shared hidden phase \(\lambda\) but **no** nonlocal dependence on settings → should not robustly exceed 2.  
2) **Pilot-on**: explicit **nonlocal configuration-space guidance** depending on **both** settings. This is a Bohmian‑flavored mechanism you can visualize on the \((\theta_A,\theta_B)\) torus.

> Tip: Run top-to-bottom, then play with the “Toggles / knobs” cell near the end.


In [ ]:
# --- Imports ---
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict
import imageio.v2 as imageio

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11
np.random.seed(42)

print("✅ Imports ready")


## 1) Simulation Config
We simulate `n_cycles` of a 60 Hz fundamental, sampled at `fs`.


In [ ]:
@dataclass
class SimConfig:
    fs: float = 10000.0         # Sampling frequency [Hz]
    f0: float = 60.0            # Fundamental frequency [Hz]
    n_cycles: int = 12          # Cycles per trial
    k_sogi: float = np.sqrt(2)  # SOGI gain
    n_trials: int = 800         # CHSH trials per setting

    @property
    def omega0(self): return 2 * np.pi * self.f0
    @property
    def T(self): return self.n_cycles / self.f0
    @property
    def dt(self): return 1.0 / self.fs
    @property
    def t(self): return np.arange(0, self.T, self.dt)

config = SimConfig()
print(f"Config: fs={config.fs:.0f} Hz, f0={config.f0:.0f} Hz, T={config.T:.3f} s, samples/trial={len(config.t)}")


## 2) Signal Generation + Clarke Transform

We work with αβ as a complex vector:
\[
V_{\alpha\beta}(t) = q_+ e^{+j\omega t} + q_- e^{-j\omega t}
\]
Then use inverse Clarke to produce abc waveforms for visualization.


In [ ]:
def generate_3phase_signals(q_plus: complex, q_minus: complex, t: np.ndarray, omega0: float):
    # Generate abc waveforms from constant sequence phasors q+ and q-
    V_ab = q_plus * np.exp(1j * omega0 * t) + q_minus * np.exp(-1j * omega0 * t)
    v_alpha, v_beta = np.real(V_ab), np.imag(V_ab)
    # Inverse Clarke (scaling not critical here)
    v_a = v_alpha
    v_b = -0.5 * v_alpha + np.sqrt(3)/2 * v_beta
    v_c = -0.5 * v_alpha - np.sqrt(3)/2 * v_beta
    return v_a, v_b, v_c

def clarke_transform(v_a: np.ndarray, v_b: np.ndarray, v_c: np.ndarray):
    # Clarke transform: abc -> alpha-beta (amplitude invariant)
    K = 2/3
    v_alpha = K * (v_a - 0.5*v_b - 0.5*v_c)
    v_beta  = K * (np.sqrt(3)/2*v_b - np.sqrt(3)/2*v_c)
    return v_alpha, v_beta

# quick sanity demo
q_plus_demo = 1.0 * np.exp(1j*np.pi/6)
q_minus_demo = 0.6 * np.exp(-1j*np.pi/4)
va, vb, vc = generate_3phase_signals(q_plus_demo, q_minus_demo, config.t, config.omega0)
v_alpha, v_beta = clarke_transform(va, vb, vc)

t_ms = config.t * 1000
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(t_ms[:900], va[:900], lw=0.8, label="Va")
ax[0].plot(t_ms[:900], vb[:900], lw=0.8, label="Vb")
ax[0].plot(t_ms[:900], vc[:900], lw=0.8, label="Vc")
ax[0].set_title("abc (snippet)"); ax[0].grid(True, alpha=0.25); ax[0].legend(frameon=False)

ax[1].plot(v_alpha[:900], v_beta[:900], lw=0.6)
ax[1].set_title("αβ trajectory (snippet)"); ax[1].axis("equal"); ax[1].grid(True, alpha=0.25)
plt.tight_layout(); plt.show()


## 3) DSOGI + Phasor Extraction

DSOGI separates + and − sequences in the stationary αβ frame. Then we demodulate to recover constant phasors \(q_+\) and \(q_-\).


In [ ]:
class SOGI:
    def __init__(self, omega0: float, k: float, dt: float):
        self.omega0 = omega0
        self.k = k
        self.dt = dt

    def process(self, v: np.ndarray):
        n = len(v)
        v_prime = np.zeros(n)
        qv_prime = np.zeros(n)
        x1, x2 = 0.0, 0.0
        for i in range(n):
            err = v[i] - x1
            x1 += (self.k * self.omega0 * err - self.omega0 * x2) * self.dt
            x2 += self.omega0 * x1 * self.dt
            v_prime[i] = x1
            qv_prime[i] = x2
        return v_prime, qv_prime

class DSOGI:
    def __init__(self, omega0: float, k: float, dt: float):
        self.sogi_a = SOGI(omega0, k, dt)
        self.sogi_b = SOGI(omega0, k, dt)

    def process(self, v_alpha: np.ndarray, v_beta: np.ndarray) -> Dict[str, np.ndarray]:
        va_p, qva = self.sogi_a.process(v_alpha)
        vb_p, qvb = self.sogi_b.process(v_beta)
        return {
            "alpha_plus":  0.5*(va_p - qvb),
            "beta_plus":   0.5*(qva + vb_p),
            "alpha_minus": 0.5*(va_p + qvb),
            "beta_minus":  0.5*(-qva + vb_p),
        }

def extract_phasor(v_alpha: np.ndarray, v_beta: np.ndarray, t: np.ndarray, omega0: float, direction: str = "positive"):
    # Demodulate and average to recover complex phasor for + or - rotating component
    v_complex = v_alpha + 1j*v_beta
    if direction == "positive":
        demod = v_complex * np.exp(-1j * omega0 * t)
    else:
        demod = v_complex * np.exp(+1j * omega0 * t)
    return np.mean(demod[len(t)//4:])

# round-trip check
dsogi = DSOGI(config.omega0, config.k_sogi, config.dt)
r = dsogi.process(v_alpha, v_beta)
q_p_rec = extract_phasor(r["alpha_plus"], r["beta_plus"], config.t, config.omega0, "positive")
q_m_rec = extract_phasor(r["alpha_minus"], r["beta_minus"], config.t, config.omega0, "negative")
print("q+ true:", q_plus_demo, "recovered:", q_p_rec)
print("q- true:", q_minus_demo, "recovered:", q_m_rec)


## 4) “Measurement Box” and Score \(S\)

Given a setting angle \(\alpha\), we compute two “channels” \(a_x,a_y\) from the recovered phasors \(q_+,q_-\).

Then:
\[
S = |a_x|^2 - |a_y|^2
\]
Interpretation: “which output port has more power?”


In [ ]:
class TimeDomainAnalyzer:
    def __init__(self, theta: float, config: SimConfig):
        # Convention: alpha = 2*theta
        self.alpha = 2 * theta
        self.config = config
        self.dsogi = DSOGI(config.omega0, config.k_sogi, config.dt)

    def measure_S(self, v_a: np.ndarray, v_b: np.ndarray, v_c: np.ndarray) -> float:
        v_alpha, v_beta = clarke_transform(v_a, v_b, v_c)
        r = self.dsogi.process(v_alpha, v_beta)
        q_p = extract_phasor(r["alpha_plus"],  r["beta_plus"],  self.config.t, self.config.omega0, "positive")
        q_m = extract_phasor(r["alpha_minus"], r["beta_minus"], self.config.t, self.config.omega0, "negative")

        ax = (np.exp(-1j*self.alpha/2)*q_p + np.exp(+1j*self.alpha/2)*q_m) / np.sqrt(2)
        ay = (np.exp(-1j*(self.alpha+np.pi)/2)*q_p + np.exp(+1j*(self.alpha+np.pi)/2)*q_m) / np.sqrt(2)

        Px, Py = np.abs(ax)**2, np.abs(ay)**2
        return float(Px - Py)


## 5) Pointer latch: score → vote

A detector-like layer: a bistable variable \(y(t)\) driven by a squashed version of \(S\).

Vote rule: \(A = \mathrm{sign}(y(T))\).


In [ ]:
class PointerLatch:
    def __init__(self, gamma=1.0, eta=2.0, dt=0.01, T_settle=1.0):
        self.gamma = gamma
        self.eta = eta
        self.dt = dt
        self.T_settle = T_settle

    def run(self, drive: float, y0: float = 0.0):
        n = int(self.T_settle / self.dt)
        y = float(y0)
        trace = np.zeros(n, dtype=float)
        for k in range(n):
            y_dot = -self.gamma * (y * (y*y - 1.0)) + self.eta * drive
            y += self.dt * y_dot
            trace[k] = y
        return y, trace

    @staticmethod
    def vote(y_end: float) -> int:
        return +1 if y_end >= 0 else -1

class AnalyzerWithPointer(TimeDomainAnalyzer):
    def __init__(self, theta: float, config: SimConfig, pointer: PointerLatch):
        super().__init__(theta, config)
        self.pointer = pointer

    def measure_vote(self, v_a, v_b, v_c, y0=0.0):
        S = self.measure_S(v_a, v_b, v_c)
        drive = np.tanh(S)  # squash huge values, preserve sign
        y_end, y_trace = self.pointer.run(drive, y0=y0)
        A = self.pointer.vote(y_end)
        return A, S, y_trace


## 6) Source models

### 6.1 Local-only source (baseline)
A shared hidden phase \(\lambda\) seeds both Alice and Bob sequences, but does **not** depend on \((a,b)\).

### 6.2 Pilot-on source (explicitly nonlocal)
A joint guidance term is defined on the unwrapped torus \((\theta_A,\theta_B)\), with pull derived from pilot phase gradient.


In [ ]:
# --- Pilot field helpers (torus story) ---
eps = 0.30

def psi_mag2(d):
    return (np.cos(d)**2 + (eps**2)*(np.sin(d)**2))

def dphi_dd(d):
    denom = (np.cos(d)**2 + (eps**2)*(np.sin(d)**2))
    return eps / denom

def evolve_rotors(thetaA0, thetaB0, a_setting, b_setting, steps=70, dt=0.03, omega=1.0, k_pull=0.55):
    # Evolve two angles with equal/opposite pull; depends on BOTH settings (nonlocal guidance story)
    thetaA, thetaB = float(thetaA0), float(thetaB0)
    pull_hist = np.zeros(steps)
    for k in range(steps):
        d = (thetaA - thetaB) - 2*(a_setting - b_setting)
        pull = dphi_dd(d)
        thetaA = (thetaA + dt*(omega + k_pull*pull)) % (2*np.pi)
        thetaB = (thetaB + dt*(omega - k_pull*pull)) % (2*np.pi)
        pull_hist[k] = pull
    return thetaA, thetaB, pull_hist

def generate_pair_local(config: SimConfig):
    # Local-only: shared hidden phase lambda; no dependence on settings
    lam = np.random.uniform(0, 2*np.pi)
    q_pA, q_mA = np.exp(1j*lam), np.exp(-1j*lam)
    q_pB, q_mB = -np.exp(1j*lam), np.exp(-1j*lam)
    alice = generate_3phase_signals(q_pA, q_mA, config.t, config.omega0)
    bob   = generate_3phase_signals(q_pB, q_mB, config.t, config.omega0)
    aux = dict(lam=lam)
    return alice, bob, aux

def generate_pair_pilot(config: SimConfig, a_setting: float, b_setting: float):
    # Pilot-on: update thetaA/thetaB via nonlocal guidance (depends on BOTH settings), then define q+/- from them
    lam = np.random.uniform(0, 2*np.pi)
    thetaA0 = (lam + 0.9) % (2*np.pi)
    thetaB0 = (lam + np.pi + 4.8) % (2*np.pi)

    thetaA, thetaB, pull_hist = evolve_rotors(
        thetaA0, thetaB0, a_setting=a_setting, b_setting=b_setting,
        steps=70, dt=0.03, omega=1.0, k_pull=0.55
    )

    q_pA, q_mA = np.exp(1j*thetaA), np.exp(-1j*thetaA)
    q_pB, q_mB = -np.exp(1j*thetaB), np.exp(-1j*thetaB)

    alice = generate_3phase_signals(q_pA, q_mA, config.t, config.omega0)
    bob   = generate_3phase_signals(q_pB, q_mB, config.t, config.omega0)

    aux = dict(lam=lam, thetaA=thetaA, thetaB=thetaB, pull_hist=pull_hist)
    return alice, bob, aux


## 7) CHSH runner (strict votes)

Settings (standard CHSH choice):
- Alice: \(\alpha\in\{0^\circ, 90^\circ\}\) → `theta_A=[0, π/4]` (since α=2θ)  
- Bob: \(\beta\in\{45^\circ, -45^\circ\}\) → `theta_B=[π/8, -π/8]`

Strict correlator:
\[
E(a,b)=\langle A\cdot B\rangle,\quad S = |E_{00}+E_{01}+E_{10}-E_{11}|
\]


In [ ]:
theta_A = [0.0, np.pi/4]      # -> alpha 0°, 90°
theta_B = [np.pi/8, -np.pi/8] # -> beta  45°, -45°

def run_chsh(config: SimConfig, pilot_on: bool, pointer: PointerLatch, verbose=True):
    E = np.zeros((2,2), dtype=float)

    for i, tA in enumerate(theta_A):
        for j, tB in enumerate(theta_B):
            a_setting = 2*tA
            b_setting = 2*tB

            alice_ana = AnalyzerWithPointer(tA, config, pointer)
            bob_ana   = AnalyzerWithPointer(tB, config, pointer)

            A_vals = np.zeros(config.n_trials, dtype=int)
            B_vals = np.zeros(config.n_trials, dtype=int)

            for k in range(config.n_trials):
                if pilot_on:
                    alice_sig, bob_sig, _ = generate_pair_pilot(config, a_setting, b_setting)
                else:
                    alice_sig, bob_sig, _ = generate_pair_local(config)

                A, SA, _ = alice_ana.measure_vote(*alice_sig)
                B, SB, _ = bob_ana.measure_vote(*bob_sig)
                A_vals[k] = A
                B_vals[k] = B

            E[i,j] = np.mean(A_vals * B_vals)
            if verbose:
                print(f"E({np.rad2deg(a_setting):3.0f}°, {np.rad2deg(b_setting):4.0f}°) = {E[i,j]:+.4f}")

    S = abs(E[0,0] + E[0,1] + E[1,0] - E[1,1])
    return E, S

pointer = PointerLatch(gamma=1.0, eta=2.0, dt=0.01, T_settle=1.0)

print("=== Local-only (pilot OFF) ===")
E0, S0 = run_chsh(config, pilot_on=False, pointer=pointer, verbose=True)
print(f"CHSH S = {S0:.4f}  (classical bound 2.000, quantum max 2.828)\n")

print("=== Pilot ON (explicitly nonlocal guidance) ===")
E1, S1 = run_chsh(config, pilot_on=True, pointer=pointer, verbose=True)
print(f"CHSH S = {S1:.4f}  (classical bound 2.000, quantum max 2.828)")


## 8) Summary plots


In [ ]:
def plot_summary(E, S, title):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    im = axes[0].imshow(E, vmin=-1, vmax=1, cmap="RdBu")
    axes[0].set_xticks([0,1]); axes[0].set_xticklabels(["45°","-45°"])
    axes[0].set_yticks([0,1]); axes[0].set_yticklabels(["0°","90°"])
    axes[0].set_xlabel("Bob β"); axes[0].set_ylabel("Alice α")
    axes[0].set_title("E(a,b) matrix")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, f"{E[i,j]:+.3f}", ha="center", va="center", color="white", fontweight="bold")
    plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    vals = [S, 2*np.sqrt(2), 2.0]
    bars = axes[1].bar(["Measured", "Quantum\nBound", "Classical\nBound"], vals, alpha=0.75)
    axes[1].axhline(2.0, ls="--", alpha=0.5)
    axes[1].set_ylim(0, 3.2)
    axes[1].set_ylabel("S")
    axes[1].set_title("CHSH parameter")
    for bar, v in zip(bars, vals):
        axes[1].text(bar.get_x()+bar.get_width()/2, v+0.05, f"{v:.3f}", ha="center")

    angles = np.linspace(-np.pi, np.pi, 200)
    axes[2].plot(angles, -np.cos(angles), lw=2, label="Reference: -cos(Δ)")
    meas_angles = [2*theta_A[i] - 2*theta_B[j] for i in range(2) for j in range(2)]
    meas_E = [E[i,j] for i in range(2) for j in range(2)]
    axes[2].plot(meas_angles, meas_E, "o", ms=9, label="Measured points")
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlabel("Δ = α - β (rad)")
    axes[2].set_ylabel("E")
    axes[2].set_title("Correlation vs Δ")
    axes[2].legend(frameon=False)

    fig.suptitle(title, y=1.02, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_summary(E0, S0, "Local-only source (pilot OFF)")
plot_summary(E1, S1, "Pilot-on source (explicitly nonlocal)")


## 9) One-trial “GIF-style” explanatory panel (+ optional GIF export)

2×2 layout:
- **Top:** Alice abc, Bob abc  
- **Bottom-left:** torus heatmap \(|\Psi(\theta_A,\theta_B)|^2\) with dot at \((\theta_A,\theta_B)\)  
- **Bottom-right:** pointer traces \(y_A(t),y_B(t)\) that become votes  

Set `make_gif=True` to write a short GIF.


In [ ]:
# Precompute torus grid once
n_grid = 140
th = np.linspace(0, 2*np.pi, n_grid, endpoint=False)
THA, THB = np.meshgrid(th, th, indexing="xy")

def render_one_trial(pilot_on=True, tA=0.0, tB=np.pi/8, make_gif=False, gif_path="chsh_pilot_pointer.gif"):
    a_setting = 2*tA
    b_setting = 2*tB

    pointer = PointerLatch(gamma=1.0, eta=2.0, dt=0.01, T_settle=1.0)
    alice_ana = AnalyzerWithPointer(tA, config, pointer)
    bob_ana   = AnalyzerWithPointer(tB, config, pointer)

    frames = []
    n_frames = 20

    if pilot_on:
        alice_sig, bob_sig, aux = generate_pair_pilot(config, a_setting, b_setting)
        thetaA = aux["thetaA"]; thetaB = aux["thetaB"]
        pull_hist = aux["pull_hist"]
    else:
        alice_sig, bob_sig, aux = generate_pair_local(config)
        thetaA = aux["lam"]
        thetaB = (aux["lam"] + np.pi) % (2*np.pi)
        pull_hist = np.zeros(70)

    MAG2 = psi_mag2((THA - THB) - 2*(a_setting - b_setting))

    A, SA, yA_trace = alice_ana.measure_vote(*alice_sig)
    B, SB, yB_trace = bob_ana.measure_vote(*bob_sig)
    voteA = "+1" if A > 0 else "-1"
    voteB = "+1" if B > 0 else "-1"

    tp = np.linspace(0, pointer.T_settle, len(yA_trace))
    frame_ids = np.linspace(0, len(pull_hist)-1, n_frames).astype(int)

    for fk in frame_ids:
        pull_val = pull_hist[fk]

        fig = plt.figure(figsize=(9.6, 5.4))
        gs = fig.add_gridspec(2, 2)
        axA = fig.add_subplot(gs[0,0])
        axB = fig.add_subplot(gs[0,1])
        axH = fig.add_subplot(gs[1,0])
        axP = fig.add_subplot(gs[1,1])

        # waveforms (snippet)
        t_ms = config.t * 1000
        sl = slice(0, min(1400, len(config.t)))

        va, vb, vc = alice_sig
        axA.plot(t_ms[sl], va[sl], lw=0.8, label="Va")
        axA.plot(t_ms[sl], vb[sl], lw=0.8, label="Vb")
        axA.plot(t_ms[sl], vc[sl], lw=0.8, label="Vc")
        axA.set_title("Alice abc"); axA.grid(True, alpha=0.25)
        axA.set_xlabel("ms"); axA.legend(frameon=False, fontsize=8, loc="upper right")

        va, vb, vc = bob_sig
        axB.plot(t_ms[sl], va[sl], lw=0.8, label="Va")
        axB.plot(t_ms[sl], vb[sl], lw=0.8, label="Vb")
        axB.plot(t_ms[sl], vc[sl], lw=0.8, label="Vc")
        axB.set_title("Bob abc"); axB.grid(True, alpha=0.25)
        axB.set_xlabel("ms"); axB.legend(frameon=False, fontsize=8, loc="upper right")

        # torus heatmap
        axH.imshow(MAG2, origin="lower", extent=[0,2*np.pi,0,2*np.pi], aspect="auto")
        axH.plot([thetaA], [thetaB], marker="o", markersize=6)
        axH.set_xlabel("θA"); axH.set_ylabel("θB")
        axH.set_title("|Ψ(θA,θB)|²  (torus unwrapped)")

        # pointer traces (reveal gradually)
        frac = fk / max(1, (len(pull_hist)-1))
        k_trace = int(frac * (len(tp)-1)) + 1
        axP.plot(tp[:k_trace], yA_trace[:k_trace], lw=1.2, label="yA")
        axP.plot(tp[:k_trace], yB_trace[:k_trace], lw=1.2, label="yB")
        axP.axhline(0, ls="--", lw=1)
        axP.set_xlim(0, pointer.T_settle); axP.set_ylim(-2.0, 2.0)
        axP.grid(True, alpha=0.25)
        axP.set_title("Pointers (votes)")
        axP.set_xlabel("settle time (s)")
        axP.legend(frameon=False, fontsize=8, loc="upper right")

        fig.subplots_adjust(top=0.86, hspace=0.55, wspace=0.28)
        fig.suptitle(
            f"pilot_on={pilot_on}  pull≈{pull_val:.2f}  votes≈({voteA},{voteB})  S_A={SA:+.2f}  S_B={SB:+.2f}",
            y=0.98, fontsize=11
        )

        if make_gif:
            fig.canvas.draw()
            img = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
            img = img.reshape(fig.canvas.get_width_height()[::-1] + (4,))[:, :, :3]
            frames.append(img)
            plt.close(fig)
        else:
            plt.show()
            break

    if make_gif:
        imageio.mimsave(gif_path, frames, duration=0.12)
        print("Wrote GIF:", gif_path)

render_one_trial(pilot_on=True, tA=0.0, tB=np.pi/8, make_gif=False)


## 10) Toggles / knobs

Try tweaking:
- pointer: `PointerLatch(gamma=..., eta=..., T_settle=...)`
- pilot: `eps`, `k_pull`, rotor steps/dt inside `evolve_rotors`
- trials: `config.n_trials`
